# House Sale Price Prediction Using Machine Learning

## Project Objective

The objective of this project is to build a complete machine-learning pipeline
that predicts the actual sale price of a residential property using its
structural, location, quality, condition, and sale-related features.

The project includes:

- Machine-learning introduction
- Exploratory Data Analysis
- Missing-value and duplicate analysis
- Outlier investigation
- Important visualizations
- Data preprocessing
- Linear Regression mathematics
- Model training
- Regression evaluation
- Actual versus predicted price analysis
- Residual analysis
- User-input house-price prediction
- A separate classification section for confusion matrix, accuracy, precision,
  recall, F1 score, and confidence probability

In [ ]:
# Cell 2: Verify Selected Python Environment

import sys
print("Python version:", sys.version)
print("Python executable:", sys.executable)


In [ ]:
# Cell 3: Verify Dataset Files

from pathlib import Path

# Find the folder in which the notebook is currently running
current_folder = Path.cwd()

# If the notebook is running inside the "notebooks" folder,
# move one level upward to reach the main project folder
project_folder = (
    current_folder.parent
    if current_folder.name.lower() == "notebooks"
    else current_folder
)

# Create paths to the data folder and required files
data_folder = project_folder / "data"
train_file = data_folder / "train.csv"
description_file = data_folder / "data_description.txt"

# Display paths and verify whether the files exist
print("Current folder:", current_folder)
print("Project folder:", project_folder)
print("Data folder:", data_folder)
print("train.csv exists:", train_file.exists())
print("data_description.txt exists:", description_file.exists())

# Cell 4: Introduction to Machine Learning

Machine learning is a branch of artificial intelligence in which a computer
learns patterns from previous data and uses those patterns to make predictions
or decisions about new data.

In traditional programming, a programmer manually writes rules:

Data + Rules → Output

In machine learning, the model learns the rules from examples:

Historical Data + Correct Outputs → Learned Model

In this project, the historical data contains information about previously sold
houses. The model will learn the relationship between house characteristics and
their actual sale prices.

The input features may include:

- Living area
- Number of bedrooms
- Number of bathrooms
- Neighborhood
- Construction year
- Overall quality
- Garage capacity
- Basement area

The target variable is:

- `SalePrice`

After training, the model will receive the characteristics of a new house and
predict its possible sale price.

## Type of Machine Learning

This project uses supervised learning because the training dataset contains both:

- Input features describing each house
- The correct sale price of each house

## Regression Problem

House-price prediction is a regression problem because the model predicts a
continuous numerical value, such as:

- $150,000
- $275,500
- $410,000

A classification model would instead predict a category such as cheap, medium,
or expensive.

In [ ]:
#Cell 5: Import Data-Analysis Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


print("ALL DATA-ANALYSIS LIBRARIES IMPORTED SUCCEUSSFULLY.")


In [ ]:
# Cell 6: Configure Visualization Display

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Visualization and display settings configured.")

In [ ]:
# Cell 7: Load the Original Housing Dataset

housing_df = pd.read_csv(
    train_file,
    keep_default_na=False,
    na_values=["NA"]
)

print("Dataset loaded successfully.")
print("Number of rows:", housing_df.shape[0])
print("Number of columns:", housing_df.shape[1])

In [ ]:
#Cell 8: Preview the First Five Houses

housing_df.head()

In [ ]:
# Cell 9: Preview the Last Five Houses

housing_df.tail()

In [ ]:
#Cell 10: Display Random House Record

housing_df.sample(5, random_state=42)

In [ ]:
# Cell 11: Confirm Dataset Dimensions

number_of_rows, number_of_columns = housing_df.shape

print("Number of house records:", number_of_rows)
print("Total number of columns:", number_of_columns)

In [ ]:
#Cell 12: Identify Features, Target and Identifiers

identifier_column = "Id"
target_column = "SalePrice"

candidate_feature_column = [
    column
    for column in housing_df.columns
    if column not in [identifier_column, target_column]

]

print("Identifiers column:", identifier_column)
print("Target Column:", target_column)
print("Number of candidate input features:", len(candidate_feature_column))

In [ ]:
# Cell 13: Display All Column Names

for column_number, column_name in enumerate(housing_df.columns, start=1):
    print(f"{column_number}. {column_name}")

In [ ]:
# Cell 14: Inspect Dataset Information

housing_df.info()

# Cell 15: Initial Dataset Structure Findings

The dataset contains 1,460 residential property records and 81 columns.

- Each row represents one sold property.
- `SalePrice` is the target variable that the model will predict.
- `Id` is a record identifier and is not a meaningful house characteristic.
- The remaining 79 columns are candidate input features.
- The dataset contains both numerical and categorical information.
- Some columns have fewer than 1,460 non-null values, indicating missing data.
- No cleaning or modification has been performed at this stage.

In [ ]:
# Cell 16: Count Columns by Technical Data Type

data_type_counts = housing_df.dtypes.value_counts()

print("Number of columns by Pandas data type:")
print(data_type_counts)

In [ ]:
# Cell 17: Separate Columns Using Pandas Data Types

technical_numerical_columns = housing_df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

technical_categorical_columns = housing_df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Technically numerical columns:", len(technical_numerical_columns))
print("Technically categorical columns:", len(technical_categorical_columns))

In [ ]:
# Cell 18: Display Technically Numerical Columns

for column_number, column_name in enumerate(
    technical_numerical_columns,
    start=1
):
    print(f"{column_number}. {column_name}")

In [ ]:
# Cell 19: Review Important Numerically Stored Columns

semantic_review = pd.DataFrame(
    {
        "Column": [
            "MSSubClass",
            "MoSold",
            "OverallQual",
            "OverallCond",
            "YearBuilt",
            "YrSold",
        ],
        "Stored_Type": [
            str(housing_df["MSSubClass"].dtype),
            str(housing_df["MoSold"].dtype),
            str(housing_df["OverallQual"].dtype),
            str(housing_df["OverallCond"].dtype),
            str(housing_df["YearBuilt"].dtype),
            str(housing_df["YrSold"].dtype),
        ],
        "Real_Meaning": [
            "Dwelling type code",
            "Month of sale",
            "Ordered house quality score",
            "Ordered house condition score",
            "Construction year",
            "Sale year",
        ],
        "Semantic_Type": [
            "Nominal categorical",
            "Calendar categorical",
            "Ordinal numerical",
            "Ordinal numerical",
            "Numerical year",
            "Numerical year",
        ],
    }
)

semantic_review


In [ ]:
# Cell 20: Define Numerical and Categorical Feature Groups

identifier_column = "Id"
target_column = "SalePrice"

# Numerical codes that logically represent categories
numeric_but_categorical = [
    "MSSubClass",
    "MoSold",
]

# Object columns are categorical by default
categorical_feature_columns = (
    technical_categorical_columns
    + numeric_but_categorical
)

# Remove categorical codes, identifier, and target
# from the numerical feature list
numerical_feature_columns = [
    column
    for column in technical_numerical_columns
    if column not in (
        numeric_but_categorical
        + [identifier_column, target_column]
    )
]

print("Numerical input features:", len(numerical_feature_columns))
print("Categorical input features:", len(categorical_feature_columns))
print("Target column:", target_column)
print("Identifier column:", identifier_column)

In [ ]:
# Cell 21: Verify Feature Classification

classified_columns = (
    numerical_feature_columns
    + categorical_feature_columns
    + [identifier_column, target_column]
)

unclassified_columns = [
    column
    for column in housing_df.columns
    if column not in classified_columns
]

duplicated_classifications = [
    column
    for column in classified_columns
    if classified_columns.count(column) > 1
]

print("Total dataset columns:", housing_df.shape[1])
print("Total classified columns:", len(classified_columns))
print("Unclassified columns:", unclassified_columns)
print(
    "Columns assigned more than once:",
    sorted(set(duplicated_classifications))
)

# Cell 23: Missing-Value Analysis

A missing value means that a value is unavailable or represented by a missing
marker in the dataset.

However, missing values do not always mean that data was accidentally omitted.

In this housing dataset, a missing value may mean:

1. The information was not recorded.
2. The house does not contain that feature.
3. The value is genuinely unknown.

For example:

- Missing `PoolQC` may mean that the house has no pool.
- Missing `GarageType` may mean that the house has no garage.
- Missing `LotFrontage` may mean that the frontage measurement is unknown.

Therefore, missing values must be investigated before deciding how to handle
them.

In [ ]:
# Cell 24: Count Missing Values in Every Column

missing_value_counts = housing_df.isna().sum()

print(missing_value_counts)

In [ ]:
# Cell 25: Create Missing-Value Summary Table

missing_summary = pd.DataFrame(
    {
        "Missing_Count": housing_df.isna().sum(),
        "Missing_Percentage": (
            housing_df.isna().sum() / len(housing_df)
        ) * 100,
    }
)

# Keep only columns containing at least one missing value
missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
]

# Arrange columns from highest to lowest missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

# Round percentages to two decimal places
missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Percentage"].round(2)
)

missing_summary

In [ ]:
# Cell 26: Summarize Missing Values

columns_with_missing_values = missing_summary.shape[0]
total_missing_cells = housing_df.isna().sum().sum()
total_dataset_cells = housing_df.shape[0] * housing_df.shape[1]

overall_missing_percentage = (
    total_missing_cells / total_dataset_cells
) * 100

print(
    "Columns containing missing values:",
    columns_with_missing_values
)

print(
    "Total missing cells:",
    total_missing_cells
)

print(
    "Total cells in the dataset:",
    total_dataset_cells
)

print(
    "Overall missing percentage:",
    round(overall_missing_percentage, 2),
    "%"
)

In [ ]:
# Cell 27: Visualize Missing-Value Percentages

plt.figure(figsize=(12, 8))

sns.barplot(
    data=missing_summary.reset_index(),
    x="Missing_Percentage",
    y="index"
)

plt.title(
    "Percentage of Missing Values by Column",
    fontsize=14
)

plt.xlabel("Missing Values (%)")
plt.ylabel("Column Name")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 28: Classify Missing Values by Meaning

missing_value_meaning = pd.DataFrame(
    {
        "Column_Group": [
            "PoolQC",
            "MiscFeature",
            "Alley",
            "Fence",
            "FireplaceQu",
            "Garage-related columns",
            "Basement-related columns",
            "LotFrontage",
            "MasVnrType and MasVnrArea",
            "Electrical",
        ],
        "Likely_Meaning": [
            "House probably has no pool",
            "House probably has no miscellaneous feature",
            "House probably has no alley access",
            "House probably has no fence",
            "House probably has no fireplace",
            "House probably has no garage",
            "House probably has no basement",
            "Street frontage measurement is unknown",
            "Masonry veneer information is unknown",
            "Electrical-system information is unknown",
        ],
        "Missing_Type": [
            "Structural absence",
            "Structural absence",
            "Structural absence",
            "Structural absence",
            "Structural absence",
            "Structural absence",
            "Structural absence",
            "Unknown numerical value",
            "Unknown feature information",
            "Unknown categorical value",
        ],
    }
)

missing_value_meaning

# Cell 29: Missing-Value Findings

The dataset contains 19 columns with at least one missing value.

Important observations:

- `PoolQC`, `MiscFeature`, `Alley`, and `Fence` have very high missing
  percentages.
- High missingness does not automatically mean that these columns are useless.
- For many categorical housing features, a missing marker represents the
  absence of the physical feature, such as no pool, no garage, or no basement.
- `LotFrontage` contains missing numerical measurements and will require an
  appropriate numerical imputation strategy.
- `MasVnrType`, `MasVnrArea`, and `Electrical` contain a small number of
  genuinely unknown values.
- No values have been filled, deleted, or modified during this EDA stage.
- Final missing-value handling will be performed during preprocessing without
  using information from the test set.

# Cell 30: Duplicate-Record Analysis

A duplicate record occurs when the same property observation appears more than
once in the dataset.

Repeated values in an individual column are normal. For example, many houses
can have the same number of bedrooms, neighborhood, or sale price.

A duplicate problem occurs when an entire property record is repeated.

Duplicate records can:

- Give repeated properties too much influence during model training
- Cause the same information to appear in both training and testing data
- Make model evaluation appear better than it actually is

Therefore, both exact duplicate rows and repeated property records should be
investigated.

In [ ]:
# Cell 31: Check Exact Duplicate Rows

exact_duplicate_count = housing_df.duplicated().sum()

print("Number of exact duplicate rows:", exact_duplicate_count)

In [ ]:
# Cell 32: Check Identifier and Property Record Uniqueness

unique_id_count = housing_df["Id"].nunique()
duplicate_id_count = housing_df["Id"].duplicated().sum()

# Remove Id before checking whether property information is repeated
duplicate_properties_without_id = (
    housing_df
    .drop(columns=["Id"])
    .duplicated()
    .sum()
)

print("Total rows:", len(housing_df))
print("Number of unique Id values:", unique_id_count)
print("Number of duplicated Id values:", duplicate_id_count)
print(
    "Duplicate property records excluding Id:",
    duplicate_properties_without_id
)

# Cell 33: Duplicate Analysis Findings

The duplicate analysis shows that:

- The dataset contains no exact duplicate rows.
- Every `Id` value is unique.
- No duplicate property records were found after excluding the `Id` column.
- Therefore, no records need to be removed because of duplication at this
  stage.
- The `Id` column remains an identifier and will later be excluded from model
  training.

In [ ]:
# Cell 34: Examine Important Numerical Value Ranges

important_numerical_columns = [
    "SalePrice",
    "LotArea",
    "GrLivArea",
    "BedroomAbvGr",
    "FullBath",
    "HalfBath",
    "OverallQual",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "YrSold",
    "GarageCars",
    "PoolArea",
]

numerical_range_summary = (
    housing_df[important_numerical_columns]
    .agg(["count", "min", "max", "mean", "median"])
    .T
)

numerical_range_summary

In [ ]:
# Cell 35: Perform Basic Validity Checks

validity_checks = {
    "SalePrice is zero or negative":
        (housing_df["SalePrice"] <= 0).sum(),

    "LotArea is zero or negative":
        (housing_df["LotArea"] <= 0).sum(),

    "GrLivArea is zero or negative":
        (housing_df["GrLivArea"] <= 0).sum(),

    "BedroomAbvGr is negative":
        (housing_df["BedroomAbvGr"] < 0).sum(),

    "OverallQual is outside 1 to 10":
        (
            (housing_df["OverallQual"] < 1)
            | (housing_df["OverallQual"] > 10)
        ).sum(),

    "OverallCond is outside 1 to 10":
        (
            (housing_df["OverallCond"] < 1)
            | (housing_df["OverallCond"] > 10)
        ).sum(),

    "YearBuilt is after YrSold":
        (housing_df["YearBuilt"] > housing_df["YrSold"]).sum(),

    "YearRemodAdd is before YearBuilt":
        (
            housing_df["YearRemodAdd"]
            < housing_df["YearBuilt"]
        ).sum(),

    "YearRemodAdd is after YrSold":
        (
            housing_df["YearRemodAdd"]
            > housing_df["YrSold"]
        ).sum(),
}

validity_results = pd.Series(
    validity_checks,
    name="Suspicious_Record_Count"
)

validity_results

In [ ]:
# Cell 36: Display Records With Remodeling After Sale

suspicious_remodel_records = housing_df.loc[
    housing_df["YearRemodAdd"] > housing_df["YrSold"],
    [
        "Id",
        "YearBuilt",
        "YearRemodAdd",
        "YrSold",
        "SalePrice",
    ],
]

suspicious_remodel_records

# Cell 37: Invalid and Suspicious Value Findings

The initial validity checks show that:

- No house has a zero or negative `SalePrice`.
- No property has a zero or negative `LotArea`.
- No property has a zero or negative above-ground living area.
- No negative bedroom counts were found.
- `OverallQual` and `OverallCond` remain within their expected ranges.
- No house was constructed after its recorded sale year.
- One property has a remodeling year later than its sale year and requires
  further investigation.
- Several variables contain extreme values, but extreme values are not
  automatically invalid.
- No rows or values have been removed or corrected at this stage.

# Cell 38: Univariate Analysis of the Target Variable

The target variable in this project is `SalePrice`.

Before training a regression model, the target variable must be examined to
understand:

- Its minimum and maximum values
- Its average and median
- Its spread
- Its distribution
- Its skewness
- Its possible outliers

Understanding the target helps determine whether transformations or special
treatment may be useful during modeling.

In [ ]:
# Cell 39: Descriptive Statistics of SalePrice

saleprice_summary = housing_df["SalePrice"].describe()

saleprice_summary.to_frame(name="SalePrice Statistics")

In [ ]:
# Cell 40: Calculate Important SalePrice Statistics

saleprice_mean = housing_df["SalePrice"].mean()
saleprice_median = housing_df["SalePrice"].median()
saleprice_mode = housing_df["SalePrice"].mode().iloc[0]
saleprice_minimum = housing_df["SalePrice"].min()
saleprice_maximum = housing_df["SalePrice"].max()
saleprice_range = saleprice_maximum - saleprice_minimum
saleprice_standard_deviation = housing_df["SalePrice"].std()
saleprice_skewness = housing_df["SalePrice"].skew()

saleprice_statistics = pd.Series(
    {
        "Mean": saleprice_mean,
        "Median": saleprice_median,
        "Mode": saleprice_mode,
        "Minimum": saleprice_minimum,
        "Maximum": saleprice_maximum,
        "Range": saleprice_range,
        "Standard Deviation": saleprice_standard_deviation,
        "Skewness": saleprice_skewness,
    }
)

saleprice_statistics

In [ ]:
# Cell 41: Visualize the Distribution of SalePrice

plt.figure(figsize=(12, 6))

sns.histplot(
    data=housing_df,
    x="SalePrice",
    bins=40,
    kde=True
)

plt.axvline(
    saleprice_mean,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: ${saleprice_mean:,.0f}"
)

plt.axvline(
    saleprice_median,
    color="green",
    linestyle="-.",
    linewidth=2,
    label=f"Median: ${saleprice_median:,.0f}"
)

plt.title("Distribution of House Sale Prices", fontsize=14)
plt.xlabel("Sale Price ($)")
plt.ylabel("Number of Houses")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 42: Visualize Possible SalePrice Outliers

plt.figure(figsize=(12, 4))

sns.boxplot(
    data=housing_df,
    x="SalePrice"
)

plt.title("Box Plot of House Sale Prices", fontsize=14)
plt.xlabel("Sale Price ($)")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 43: Detect Possible SalePrice Outliers Using IQR

saleprice_q1 = housing_df["SalePrice"].quantile(0.25)
saleprice_q3 = housing_df["SalePrice"].quantile(0.75)

saleprice_iqr = saleprice_q3 - saleprice_q1

saleprice_lower_limit = saleprice_q1 - (1.5 * saleprice_iqr)
saleprice_upper_limit = saleprice_q3 + (1.5 * saleprice_iqr)

low_price_outliers = (
    housing_df["SalePrice"] < saleprice_lower_limit
).sum()

high_price_outliers = (
    housing_df["SalePrice"] > saleprice_upper_limit
).sum()

print("First quartile (Q1):", saleprice_q1)
print("Third quartile (Q3):", saleprice_q3)
print("Interquartile range:", saleprice_iqr)
print("Lower outlier limit:", saleprice_lower_limit)
print("Upper outlier limit:", saleprice_upper_limit)
print("Possible low-price outliers:", low_price_outliers)
print("Possible high-price outliers:", high_price_outliers)

In [ ]:

# Cell 44: Inspect the Most Expensive Houses

expensive_house_columns = [
    "Id",
    "Neighborhood",
    "OverallQual",
    "YearBuilt",
    "GrLivArea",
    "GarageCars",
    "SalePrice",
]

housing_df[
    expensive_house_columns
].sort_values(
    by="SalePrice",
    ascending=False
).head(10)

In [ ]:
# Cell 45: Create a Temporary Log-Transformed SalePrice

saleprice_log = np.log1p(housing_df["SalePrice"])

print(
    "Original SalePrice skewness:",
    round(housing_df["SalePrice"].skew(), 3)
)

print(
    "Log-transformed SalePrice skewness:",
    round(saleprice_log.skew(), 3)
)

In [ ]:
# Cell 46: Visualize Log-Transformed SalePrice

plt.figure(figsize=(12, 6))

sns.histplot(
    saleprice_log,
    bins=40,
    kde=True
)

plt.title(
    "Distribution of Log-Transformed SalePrice",
    fontsize=14
)

plt.xlabel("log(1 + SalePrice)")
plt.ylabel("Number of Houses")

plt.tight_layout()
plt.show()

# Cell 47: SalePrice Analysis Findings

The target-variable analysis produced the following findings:

- The dataset contains 1,460 available sale-price values.
- The mean sale price is approximately $180,921.
- The median sale price is approximately $163,000.
- The mean is greater than the median, indicating that expensive houses pull
  the average upward.
- The original `SalePrice` distribution is strongly right-skewed.
- The original skewness is approximately 1.88.
- The box plot and IQR method identify 61 possible high-price outliers.
- These observations must not be removed automatically because they may
  represent valid expensive properties.
- A temporary logarithmic transformation reduces the target skewness to
  approximately 0.12.
- The original target column has not been modified.
- The usefulness of target transformation will be evaluated during model
  development.

# Cell 48: Univariate Analysis of Numerical Features

Numerical univariate analysis examines one numerical feature at a time.

The purpose is to understand:

- The central value of each feature
- The spread of its values
- Its minimum and maximum
- Its distribution
- Its skewness
- The number of zero values
- Its possible outliers
- Whether the recorded values appear realistic

This analysis helps identify which numerical features may require imputation,
transformation, scaling, or further investigation during preprocessing.

No values will be removed or modified during this stage.

In [ ]:
# Cell 49: Create Numerical Feature Summary

numerical_summary = pd.DataFrame(
    index=numerical_feature_columns
)

numerical_summary["Non_Missing_Count"] = (
    housing_df[numerical_feature_columns].count()
)

numerical_summary["Missing_Count"] = (
    housing_df[numerical_feature_columns].isna().sum()
)

numerical_summary["Missing_Percentage"] = (
    housing_df[numerical_feature_columns].isna().mean() * 100
)

numerical_summary["Unique_Values"] = (
    housing_df[numerical_feature_columns].nunique()
)

numerical_summary["Mean"] = (
    housing_df[numerical_feature_columns].mean()
)

numerical_summary["Median"] = (
    housing_df[numerical_feature_columns].median()
)

numerical_summary["Standard_Deviation"] = (
    housing_df[numerical_feature_columns].std()
)

numerical_summary["Minimum"] = (
    housing_df[numerical_feature_columns].min()
)

numerical_summary["Maximum"] = (
    housing_df[numerical_feature_columns].max()
)

numerical_summary["Skewness"] = (
    housing_df[numerical_feature_columns].skew()
)

numerical_summary["Zero_Count"] = (
    housing_df[numerical_feature_columns].eq(0).sum()
)

numerical_summary["Zero_Percentage"] = (
    housing_df[numerical_feature_columns].eq(0).mean() * 100
)

numerical_summary = numerical_summary.round(2)

numerical_summary

In [ ]:
# Cell 50: Identify Highly Skewed Numerical Features

skewness_summary = numerical_summary[
    ["Skewness"]
].copy()

skewness_summary["Absolute_Skewness"] = (
    skewness_summary["Skewness"].abs()
)

skewness_summary = skewness_summary.sort_values(
    by="Absolute_Skewness",
    ascending=False
)

skewness_summary.head(15)

In [ ]:
# Cell 51: Visualize All Numerical Feature Distributions

features_per_figure = 9

for start_index in range(
    0,
    len(numerical_feature_columns),
    features_per_figure
):
    feature_batch = numerical_feature_columns[
        start_index:start_index + features_per_figure
    ]

    housing_df[feature_batch].hist(
        bins=30,
        figsize=(15, 10),
        layout=(3, 3),
        edgecolor="black"
    )

    figure_number = (
        start_index // features_per_figure
    ) + 1

    plt.suptitle(
        f"Numerical Feature Distributions — Figure {figure_number}",
        fontsize=16
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.96]
    )

    plt.show()

In [ ]:
# Cell 52: Box Plots of Important Continuous Features

key_continuous_features = [
    "LotArea",
    "GrLivArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "GarageArea",
    "WoodDeckSF",
    "OpenPorchSF",
]

figure, axes = plt.subplots(
    nrows=4,
    ncols=2,
    figsize=(14, 16)
)

axes = axes.flatten()

for axis, column in zip(
    axes,
    key_continuous_features
):
    sns.boxplot(
        data=housing_df,
        x=column,
        ax=axis
    )

    axis.set_title(
        f"Box Plot of {column}"
    )

    axis.set_xlabel(column)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 53: Visualize Important Discrete and Ordinal Features

key_discrete_features = [
    "OverallQual",
    "OverallCond",
    "BedroomAbvGr",
    "FullBath",
    "HalfBath",
    "TotRmsAbvGrd",
    "Fireplaces",
    "GarageCars",
]

figure, axes = plt.subplots(
    nrows=4,
    ncols=2,
    figsize=(14, 18)
)

axes = axes.flatten()

for axis, column in zip(
    axes,
    key_discrete_features
):
    category_order = sorted(
        housing_df[column]
        .dropna()
        .unique()
    )

    sns.countplot(
        data=housing_df,
        x=column,
        order=category_order,
        ax=axis
    )

    axis.set_title(
        f"Frequency Distribution of {column}"
    )

    axis.set_xlabel(column)
    axis.set_ylabel("Number of Houses")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 54: Identify Zero-Heavy Numerical Features

zero_value_summary = pd.DataFrame(
    {
        "Zero_Count": (
            housing_df[numerical_feature_columns]
            .eq(0)
            .sum()
        ),
        "Zero_Percentage": (
            housing_df[numerical_feature_columns]
            .eq(0)
            .mean()
            * 100
        ),
    }
)

zero_value_summary = zero_value_summary[
    zero_value_summary["Zero_Count"] > 0
]

zero_value_summary = zero_value_summary.sort_values(
    by="Zero_Percentage",
    ascending=False
)

zero_value_summary["Zero_Percentage"] = (
    zero_value_summary["Zero_Percentage"]
    .round(2)
)

zero_value_summary

In [ ]:
# Cell 55: Visualize Zero-Heavy Numerical Features

top_zero_features = (
    zero_value_summary
    .head(15)
    .reset_index()
    .rename(columns={"index": "Feature"})
)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=top_zero_features,
    x="Zero_Percentage",
    y="Feature"
)

plt.title(
    "Numerical Features With the Highest Percentage of Zero Values",
    fontsize=14
)

plt.xlabel("Zero Values (%)")
plt.ylabel("Numerical Feature")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 56: Screen Continuous Features for Possible Outliers

continuous_outlier_columns = [
    "LotFrontage",
    "LotArea",
    "GrLivArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "GarageArea",
]

outlier_results = []

for column in continuous_outlier_columns:
    feature_values = housing_df[column].dropna()

    first_quartile = feature_values.quantile(0.25)
    third_quartile = feature_values.quantile(0.75)

    interquartile_range = (
        third_quartile - first_quartile
    )

    lower_limit = (
        first_quartile
        - 1.5 * interquartile_range
    )

    upper_limit = (
        third_quartile
        + 1.5 * interquartile_range
    )

    outlier_count = (
        (
            feature_values < lower_limit
        )
        |
        (
            feature_values > upper_limit
        )
    ).sum()

    outlier_percentage = (
        outlier_count
        / len(feature_values)
    ) * 100

    outlier_results.append(
        {
            "Feature": column,
            "Q1": first_quartile,
            "Q3": third_quartile,
            "IQR": interquartile_range,
            "Lower_Limit": lower_limit,
            "Upper_Limit": upper_limit,
            "Possible_Outliers": outlier_count,
            "Outlier_Percentage": outlier_percentage,
        }
    )

continuous_outlier_summary = pd.DataFrame(
    outlier_results
)

continuous_outlier_summary = (
    continuous_outlier_summary
    .round(2)
)

continuous_outlier_summary

In [ ]:
# Cell 57: Inspect Houses With the Largest Living Areas

large_living_area_columns = [
    "Id",
    "Neighborhood",
    "OverallQual",
    "YearBuilt",
    "GrLivArea",
    "TotalBsmtSF",
    "GarageCars",
    "SalePrice",
]

largest_houses = (
    housing_df[large_living_area_columns]
    .sort_values(
        by="GrLivArea",
        ascending=False
    )
    .head(10)
)

largest_houses

# Cell 58: Numerical Feature Analysis Findings

The univariate numerical analysis produced the following observations:

- The dataset contains numerical measurements, count variables, ordered scores,
  and year-related features.
- Several area-related features are positively skewed because most properties
  have moderate values while a small number of properties are extremely large.
- Some numerical features contain a high percentage of zero values.
- In many cases, zero represents the absence of a property feature rather than
  missing or incorrect information.
- `PoolArea`, porch-area variables, miscellaneous value, and low-quality
  finished area are examples of zero-heavy features.
- Count plots show that some bedroom, bathroom, room, quality, and garage
  values are much more common than others.
- Box plots and the IQR method identify several possible outliers in continuous
  features.
- Statistically unusual values have not been removed because they may represent
  valid large or luxury properties.
- Further investigation will be performed during bivariate and multivariate
  analysis.
- No numerical feature has been transformed, scaled, filled, or deleted at
  this stage.

# Cell 59: EDA Conclusion and Preprocessing Plan

The exploratory data analysis identified numerical and categorical features,
missing values, zero-heavy features, possible outliers, and a right-skewed
target variable.

The following preprocessing decisions will be used:

- `Id` will be removed because it is only a record identifier.
- `SalePrice` will be separated as the target variable.
- The dataset will be divided into training and testing sets before fitting
  preprocessing operations.
- Missing `LotFrontage` values will be filled using the training-set median.
- Missing `GarageYrBlt` and `MasVnrArea` values will be filled with zero.
- Missing categorical values will be represented by a separate `Missing`
  category.
- Categorical features will be converted into numerical columns using
  one-hot encoding.
- Unknown categories appearing in test data will be handled safely.
- A Linear Regression model will be trained using the processed training data.

In [ ]:
# Cell 60: Separate Input Features and Target

X = housing_df.drop(
    columns=["Id", "SalePrice"]
)

y = housing_df["SalePrice"]

print("Input feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Cell 61: Split the Dataset Into Training and Testing Sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training input shape:", X_train.shape)
print("Testing input shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

In [ ]:
# Cell 62: Identify Numerical and Categorical Training Features

numeric_code_categories = [
    "MSSubClass",
    "MoSold",
]

categorical_columns = (
    X_train.select_dtypes(include=["object"])
    .columns
    .tolist()
    + numeric_code_categories
)

numerical_columns = [
    column
    for column in X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns
    if column not in categorical_columns
]

print("Numerical features:", len(numerical_columns))
print("Categorical features:", len(categorical_columns))
print(
    "Total features classified:",
    len(numerical_columns) + len(categorical_columns)
)

In [ ]:
# Cell 63: Define Numerical Missing-Value Strategies

zero_imputation_columns = [
    column
    for column in [
        "GarageYrBlt",
        "MasVnrArea",
    ]
    if column in numerical_columns
]

median_imputation_columns = [
    column
    for column in numerical_columns
    if column not in zero_imputation_columns
]

print(
    "Numerical columns filled with zero:",
    zero_imputation_columns
)

print(
    "Number of numerical columns filled with median:",
    len(median_imputation_columns)
)

In [ ]:
# Cell 64: Create the Preprocessing Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

median_numerical_pipeline = Pipeline(
    steps=[
        (
            "median_imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

zero_numerical_pipeline = Pipeline(
    steps=[
        (
            "zero_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "categorical_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "one_hot_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "median_numerical",
            median_numerical_pipeline,
            median_imputation_columns
        ),
        (
            "zero_numerical",
            zero_numerical_pipeline,
            zero_imputation_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

print("Preprocessing pipeline created successfully.")

In [ ]:
# Cell 65: Create the Complete Linear Regression Pipeline

from sklearn.linear_model import LinearRegression

linear_regression_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "linear_regression",
            LinearRegression()
        )
    ]
)

print("Complete model pipeline created successfully.")

In [ ]:
#Cell 66: Train the LIner Regression Model

linear_regression_pipeline.fit(
    X_train,
    y_train
)

print("Linear Regression model trained successfully")


In [ ]:
# Cell 67: Generate House Price Predictions

y_pred = linear_regression_pipeline.predict(
    X_test
)

print("Number of predictions:", len(y_pred))
print("First five predictions:")

for predicted_price in y_pred[:5]:
    print(f"${predicted_price:,.2f}")

In [ ]:
# Cell 68: Calculate Regression Evaluation Metrics

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    y_pred
)

regression_metrics = pd.DataFrame(
    {
        "Metric": [
            "Mean Absolute Error",
            "Mean Squared Error",
            "Root Mean Squared Error",
            "R-squared"
        ],
        "Value": [
            mae,
            mse,
            rmse,
            r2
        ]
    }
)

regression_metrics

In [ ]:
# Cell 69: Compare Actual and Predicted Sale Prices

prediction_results = pd.DataFrame(
    {
        "Actual_SalePrice": y_test.values,
        "Predicted_SalePrice": y_pred
    },
    index=y_test.index
)

prediction_results["Prediction_Error"] = (
    prediction_results["Actual_SalePrice"]
    - prediction_results["Predicted_SalePrice"]
)

prediction_results["Absolute_Error"] = (
    prediction_results["Prediction_Error"]
    .abs()
)

prediction_results.head(10).round(2)

In [ ]:
# Cell 70: Visualize Actual Versus Predicted Prices

plt.figure(figsize=(10, 7))

sns.scatterplot(
    x=y_test,
    y=y_pred,
    alpha=0.7
)

minimum_price = min(
    y_test.min(),
    y_pred.min()
)

maximum_price = max(
    y_test.max(),
    y_pred.max()
)

plt.plot(
    [minimum_price, maximum_price],
    [minimum_price, maximum_price],
    linestyle="--",
    linewidth=2,
    label="Perfect Prediction Line"
)

plt.title(
    "Actual Versus Predicted House Sale Prices",
    fontsize=14
)

plt.xlabel("Actual Sale Price ($)")
plt.ylabel("Predicted Sale Price ($)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 71: Visualize Residuals Versus Predicted Prices

residuals = y_test - y_pred

plt.figure(figsize=(10, 7))

sns.scatterplot(
    x=y_pred,
    y=residuals,
    alpha=0.7
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=2
)

plt.title(
    "Residuals Versus Predicted Sale Prices",
    fontsize=14
)

plt.xlabel("Predicted Sale Price ($)")
plt.ylabel("Residual: Actual - Predicted ($)")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 72: Visualize the Residual Distribution

plt.figure(figsize=(10, 6))

sns.histplot(
    residuals,
    bins=40,
    kde=True
)

plt.axvline(
    x=0,
    linestyle="--",
    linewidth=2
)

plt.title(
    "Distribution of Linear Regression Residuals",
    fontsize=14
)

plt.xlabel("Residual: Actual - Predicted ($)")
plt.ylabel("Number of Houses")

plt.tight_layout()
plt.show()

# Cell 73: Baseline Linear Regression Findings

The Linear Regression model was trained using 80% of the housing records and
evaluated using the remaining 20%.

The model achieved:

- Mean Absolute Error: approximately $21,139.99
- Root Mean Squared Error: approximately $31,790.77
- R-squared score: 0.8682

Interpretation:

- The Mean Absolute Error represents the average absolute difference between
  actual and predicted sale prices.
- The Root Mean Squared Error gives more importance to large prediction errors.
- The R-squared score represents the proportion of test-set sale-price
  variation explained by the model.
- The actual-versus-predicted plot shows how closely predictions follow real
  prices.
- The residual plots help determine whether prediction errors are random or
  follow a systematic pattern.
- This is the baseline Linear Regression model and may later be compared with
  a log-target version.

# Cell 74: Mathematics of Linear Regression

Linear Regression predicts a numerical value by learning a linear relationship
between input features and the target variable.

## Simple Linear Regression

When there is only one input feature, the equation is:

$$
\hat{y} = b_0 + b_1x
$$

Where:

- $\hat{y}$ is the predicted value.
- $b_0$ is the intercept.
- $b_1$ is the coefficient or slope.
- $x$ is the input feature.

For example, if living area is the only input feature:

$$
\text{Predicted Price}
=
b_0 + b_1(\text{Living Area})
$$

## Multiple Linear Regression

Our housing dataset contains multiple input features. Therefore, the model uses:

$$
\hat{y}
=
b_0
+
b_1x_1
+
b_2x_2
+
\cdots
+
b_nx_n
$$

Where:

- $\hat{y}$ is the predicted house price.
- $b_0$ is the intercept.
- $b_1, b_2, \ldots, b_n$ are learned coefficients.
- $x_1, x_2, \ldots, x_n$ are house features.
- $n$ is the total number of processed input features.

## Simplified House-Price Example

Suppose a simplified model uses only living area and overall quality:

$$
\text{Predicted Price}
=
50{,}000
+
100(\text{Living Area})
+
20{,}000(\text{Overall Quality})
$$

For a house with:

- Living area = $1{,}500$ square feet
- Overall quality = $7$

The prediction is:

$$
\text{Predicted Price}
=
50{,}000
+
100(1{,}500)
+
20{,}000(7)
$$

$$
\text{Predicted Price}
=
50{,}000
+
150{,}000
+
140{,}000
$$

$$
\text{Predicted Price}
=
340{,}000
$$

Therefore, the predicted house price is **\$340,000**.

## Residual or Prediction Error

The residual is the difference between the actual price and predicted price:

$$
\text{Residual}
=
\text{Actual Price}
-
\text{Predicted Price}
$$

Suppose the actual sale price is **\$360,000**:

$$
\text{Residual}
=
360{,}000
-
340{,}000
$$

$$
\text{Residual}
=
20{,}000
$$

A positive residual means the model predicted a value lower than the actual
price. A negative residual means the model predicted a value higher than the
actual price.

Linear Regression learns its coefficients by minimizing the total squared
prediction errors across the training data.

In [ ]:
# Cell 75: Manual Regression Error Calculation

actual_price = 360000
predicted_price = 340000

residual = actual_price - predicted_price
absolute_error = abs(residual)
squared_error = residual ** 2

print(f"Actual price: ${actual_price:,.2f}")
print(f"Predicted price: ${predicted_price:,.2f}")
print(f"Residual: ${residual:,.2f}")
print(f"Absolute error: ${absolute_error:,.2f}")
print(f"Squared error: {squared_error:,.2f} dollars²")

# Cell 76: Classification Extension

Accuracy, precision, recall, F1 score, confusion matrix, and confidence
probability are classification metrics.

They cannot correctly evaluate exact numerical house-price predictions.

To demonstrate these concepts correctly, a separate classification problem will
be created:

- Class 0: Sale price below the training-set median
- Class 1: Sale price equal to or above the training-set median

A Logistic Regression classifier will predict the price category and return a
probability-based confidence score.

In [ ]:
# Cell 77: Create House Price Classification Labels

classification_threshold = y_train.median()

y_train_class = (
    y_train >= classification_threshold
).astype(int)

y_test_class = (
    y_test >= classification_threshold
).astype(int)

print(
    "Classification threshold:",
    f"${classification_threshold:,.2f}"
)

print("\nTraining-class distribution:")
print(y_train_class.value_counts().sort_index())

print("\nTesting-class distribution:")
print(y_test_class.value_counts().sort_index())

In [ ]:
# Cell 78: Build the Logistic Regression Classification Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

classification_median_pipeline = Pipeline(
    steps=[
        (
            "median_imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "standard_scaler",
            StandardScaler()
        )
    ]
)

classification_zero_pipeline = Pipeline(
    steps=[
        (
            "zero_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        ),
        (
            "standard_scaler",
            StandardScaler()
        )
    ]
)

classification_categorical_pipeline = Pipeline(
    steps=[
        (
            "categorical_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "one_hot_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

classification_preprocessor = ColumnTransformer(
    transformers=[
        (
            "median_numerical",
            classification_median_pipeline,
            median_imputation_columns
        ),
        (
            "zero_numerical",
            classification_zero_pipeline,
            zero_imputation_columns
        ),
        (
            "categorical",
            classification_categorical_pipeline,
            categorical_columns
        )
    ]
)

classification_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            classification_preprocessor
        ),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

print("Classification pipeline created successfully.")

In [ ]:
# Cell 79: Train the House Price Classification Model

classification_pipeline.fit(
    X_train,
    y_train_class
)

y_class_pred = classification_pipeline.predict(
    X_test
)

y_class_probability = (
    classification_pipeline.predict_proba(X_test)
)

print("Classification model trained successfully.")
print("Number of class predictions:", len(y_class_pred))

In [ ]:
# Cell 80: Calculate Classification Evaluation Metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(
    y_test_class,
    y_class_pred
)

precision = precision_score(
    y_test_class,
    y_class_pred
)

recall = recall_score(
    y_test_class,
    y_class_pred
)

f1 = f1_score(
    y_test_class,
    y_class_pred
)

classification_metrics = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score"
        ],
        "Score": [
            accuracy,
            precision,
            recall,
            f1
        ]
    }
)

classification_metrics["Score"] = (
    classification_metrics["Score"].round(4)
)

classification_metrics

In [ ]:
# Cell 81: Visualize the Confusion Matrix

confusion = confusion_matrix(
    y_test_class,
    y_class_pred
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Predicted Lower Price",
        "Predicted Higher Price"
    ],
    yticklabels=[
        "Actual Lower Price",
        "Actual Higher Price"
    ]
)

plt.title("House Price Category Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()
plt.show()

# Cell 82: Mathematics of Classification Metrics

The confusion matrix contains four possible outcomes.

- **True Positive (TP):** A higher-price house is correctly predicted as higher-price.
- **True Negative (TN):** A lower-price house is correctly predicted as lower-price.
- **False Positive (FP):** A lower-price house is incorrectly predicted as higher-price.
- **False Negative (FN):** A higher-price house is incorrectly predicted as lower-price.

For our model:

- TP = 124
- TN = 152
- FP = 10
- FN = 6

## Accuracy

Accuracy measures the proportion of all predictions that are correct.

$$
\text{Accuracy}
=
\frac{TP + TN}
{TP + TN + FP + FN}
$$

Using our confusion-matrix values:

$$
\text{Accuracy}
=
\frac{124 + 152}
{124 + 152 + 10 + 6}
$$

$$
\text{Accuracy}
=
\frac{276}{292}
=
0.9452
$$

Therefore, the model correctly classified approximately **94.52%** of all
test houses.

## Precision

Precision measures how many houses predicted as higher-price were actually
higher-price.

$$
\text{Precision}
=
\frac{TP}
{TP + FP}
$$

Using our values:

$$
\text{Precision}
=
\frac{124}
{124 + 10}
$$

$$
\text{Precision}
=
\frac{124}{134}
=
0.9254
$$

Therefore, approximately **92.54%** of the houses predicted as higher-price
were actually higher-price.

## Recall

Recall measures how many actual higher-price houses were correctly identified.

$$
\text{Recall}
=
\frac{TP}
{TP + FN}
$$

Using our values:

$$
\text{Recall}
=
\frac{124}
{124 + 6}
$$

$$
\text{Recall}
=
\frac{124}{130}
=
0.9538
$$

Therefore, the model identified approximately **95.38%** of all actual
higher-price houses.

## F1 Score

The F1 score combines precision and recall into one balanced measurement.

$$
F1
=
2
\times
\frac{\text{Precision} \times \text{Recall}}
{\text{Precision} + \text{Recall}}
$$

Using our precision and recall:

$$
F1
=
2
\times
\frac{0.9254 \times 0.9538}
{0.9254 + 0.9538}
$$

$$
F1
\approx
0.9394
$$

Therefore, the model achieved an F1 score of approximately **93.94%**.

## Final Interpretation

- Accuracy evaluates overall correctness.
- Precision focuses on the reliability of higher-price predictions.
- Recall focuses on finding actual higher-price houses.
- F1 score balances precision and recall.
- These metrics evaluate the separate classification model, not the Linear
  Regression model that predicts exact house prices.

In [ ]:
# Cell 83: Display Classification Predictions With Confidence

classification_results = pd.DataFrame(
    {
        "Actual_Class": y_test_class.values,
        "Predicted_Class": y_class_pred,
        "Probability_Lower_Price": (
            y_class_probability[:, 0]
        ),
        "Probability_Higher_Price": (
            y_class_probability[:, 1]
        )
    },
    index=y_test.index
)

classification_results["Confidence_Percentage"] = (
    classification_results[
        [
            "Probability_Lower_Price",
            "Probability_Higher_Price"
        ]
    ].max(axis=1) * 100
)

classification_results.head(10).round(4)

In [ ]:
# Cell 84: Create Default Values for User Input

default_house_values = {}

for column in X_train.columns:
    if column in numerical_columns:
        default_house_values[column] = (
            X_train[column].median()
        )
    else:
        available_mode = (
            X_train[column]
            .mode(dropna=True)
        )

        if len(available_mode) > 0:
            default_house_values[column] = (
                available_mode.iloc[0]
            )
        else:
            default_house_values[column] = np.nan

default_house = pd.DataFrame(
    [default_house_values],
    columns=X_train.columns
)

print("Default house template created.")
print("Number of input features:", default_house.shape[1])

In [ ]:
# Cell 85: Interactive House Price Prediction Function

def read_numeric_input(message, minimum=None, maximum=None):
    while True:
        try:
            value = float(input(message))

            if minimum is not None and value < minimum:
                print(f"Value must be at least {minimum}.")
                continue

            if maximum is not None and value > maximum:
                print(f"Value must not exceed {maximum}.")
                continue

            return value

        except ValueError:
            print("Please enter a valid numerical value.")


def predict_new_house():
    new_house = default_house.copy()

    print("Enter the important characteristics of the house.\n")

    new_house.loc[0, "OverallQual"] = read_numeric_input(
        "Overall quality (1-10): ",
        minimum=1,
        maximum=10
    )

    new_house.loc[0, "GrLivArea"] = read_numeric_input(
        "Above-ground living area in square feet: ",
        minimum=1
    )

    new_house.loc[0, "YearBuilt"] = read_numeric_input(
        "Construction year: ",
        minimum=1800,
        maximum=2026
    )

    new_house.loc[0, "TotalBsmtSF"] = read_numeric_input(
        "Total basement area in square feet: ",
        minimum=0
    )

    new_house.loc[0, "GarageCars"] = read_numeric_input(
        "Garage capacity in cars: ",
        minimum=0
    )

    new_house.loc[0, "FullBath"] = read_numeric_input(
        "Number of full bathrooms: ",
        minimum=0
    )

    new_house.loc[0, "BedroomAbvGr"] = read_numeric_input(
        "Number of bedrooms above ground: ",
        minimum=0
    )

    valid_neighborhoods = sorted(
        X_train["Neighborhood"]
        .dropna()
        .unique()
    )

    print("\nAvailable neighborhoods:")
    print(", ".join(valid_neighborhoods))

    neighborhood_input = input(
        "\nEnter neighborhood exactly as shown: "
    ).strip()

    if neighborhood_input in valid_neighborhoods:
        new_house.loc[0, "Neighborhood"] = (
            neighborhood_input
        )
    else:
        default_neighborhood = (
            default_house.loc[0, "Neighborhood"]
        )

        print(
            "Neighborhood not recognized."
            f" Using default: {default_neighborhood}"
        )

    predicted_sale_price = (
        linear_regression_pipeline
        .predict(new_house)[0]
    )

    predicted_category = (
        classification_pipeline
        .predict(new_house)[0]
    )

    category_probabilities = (
        classification_pipeline
        .predict_proba(new_house)[0]
    )

    confidence_percentage = (
        category_probabilities.max() * 100
    )

    if predicted_category == 1:
        category_name = "Higher-price category"
    else:
        category_name = "Lower-price category"

    lower_estimate = max(
        0,
        predicted_sale_price - mae
    )

    upper_estimate = (
        predicted_sale_price + mae
    )

    print("\nPrediction Results")
    print("------------------")

    print(
        "Predicted sale price:",
        f"${predicted_sale_price:,.2f}"
    )

    print(
        "Approximate MAE-based range:",
        f"${lower_estimate:,.2f}",
        "to",
        f"${upper_estimate:,.2f}"
    )

    print("Predicted category:", category_name)

    print(
        "Classification confidence:",
        f"{confidence_percentage:.2f}%"
    )

In [ ]:
# Cell 86: Run the Interactive Prediction System

predict_new_house()

In [ ]:
# Cell 87: Save the Trained Models

import joblib

linear_model_file = (
    project_folder
    / "models"
    / "linear_regression_pipeline.joblib"
)

classification_model_file = (
    project_folder
    / "models"
    / "classification_pipeline.joblib"
)

joblib.dump(
    linear_regression_pipeline,
    linear_model_file
)

joblib.dump(
    classification_pipeline,
    classification_model_file
)

print("Linear Regression model saved to:")
print(linear_model_file)

print("\nClassification model saved to:")
print(classification_model_file)

# Cell 88: Final Verified Machine-Learning Pipeline

The completed project follows this pipeline:

1. Define the house-price prediction problem
2. Load the original uncleaned housing dataset
3. Inspect rows, columns, features, and target
4. Identify numerical and categorical variables
5. Analyze missing values
6. Check duplicate records
7. Check invalid and suspicious values
8. Analyze the target distribution
9. Visualize important numerical distributions and outliers
10. Separate input features and target
11. Split the dataset into training and testing sets
12. Learn preprocessing only from training data
13. Impute missing numerical and categorical values
14. One-hot encode categorical features
15. Train a Linear Regression model
16. Generate predictions for unseen test houses
17. Evaluate using MAE, MSE, RMSE, and R-squared
18. Analyze actual-versus-predicted prices and residuals
19. Train a separate classification model
20. Evaluate classification using confusion matrix, accuracy, precision, recall,
    and F1 score
21. Produce a probability-based classification confidence score
22. Accept user input and predict a new house price
23. Save the complete trained pipelines